# Notebook 12 — Implement the same team in CrewAI

This notebook reuses the AgentOps checkout conversion incident from the previous lessons and maps it to CrewAI's mental model: **Agents + Tasks + Crew**. CrewAI is helpful pedagogically because it makes the project-management shape of a multi-agent system very visible: roles own work, tasks describe deliverables, and the crew runs the collaboration plan.

The executable cells use a deterministic teaching double so the notebook runs without API keys. The CrewAI-shaped code is included as a reference pattern for learners who want to port the same scenario to the real framework.

References: [CrewAI docs](https://docs.crewai.com/), [CrewAI agents](https://docs.crewai.com/v1.15.10/en/concepts/agents), [CrewAI crews](https://docs.crewai.com/v1.15.6/en/concepts/crews), [CrewAI processes](https://docs.crewai.com/v1.15.5/en/concepts/processes), and [CrewAI GitHub](https://github.com/crewAIInc/crewAI).

## Scenario

Checkout conversion has fallen 35% in Europe. There is no obvious outage. The team needs to inspect telemetry, deployment history, customer impact, and then produce a remediation plan.

```mermaid
flowchart TD
    A["Observability Agent"] --> T1["metrics_task: inspect telemetry"]
    B["Deployment Agent"] --> T2["deployment_task: inspect releases"]
    C["Customer Impact Agent"] --> T3["customer_task: inspect segments"]
    T1 --> T4["analysis_task"]
    T2 --> T4
    T3 --> T4
    D["Incident Commander"] --> T4
    T4 --> P["Incident plan"]
```

The important design lesson is not that CrewAI is always better. It is that a **task graph with named owners** can make collaboration easier to read and review.

In [ ]:
from pathlib import Path
import sys

repo = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo / "labs") not in sys.path:
    sys.path.insert(0, str(repo / "labs"))

from agentops_lab.crewai_team import kickoff_crew

result = kickoff_crew()
result["mental_model"], result["final_plan"][:180]


## Inspect the crew trace

Each task has an owner and the final analysis task depends on the specialist outputs. That makes provenance much clearer than a single long transcript where one agent silently changes focus.

In [ ]:
for event in result["trace"]:
    print(f"{event['task']} | {event['agent_role']} | context={event['uses_context']}")
    print(f"  {event['output'][:140]}...\n")


## CrewAI-shaped implementation

A real CrewAI version would use `Agent`, `Task`, and `Crew` objects. Keep tools narrow and read-only until a deterministic policy layer authorizes side effects.

```python
from crewai import Agent, Task, Crew

observability = Agent(
    role="Observability Engineer",
    goal="Find operational evidence explaining the incident",
    backstory="Expert in logs, metrics, and service health.",
)

deployment = Agent(
    role="Release Engineer",
    goal="Determine whether recent deployments contributed",
    backstory="Expert in deployments and rollback history.",
)

analyst = Agent(
    role="Incident Commander",
    goal="Determine the most likely cause and recommend action",
    backstory="Synthesizes evidence from engineering specialists.",
)

metrics_task = Task(description="Investigate service telemetry.", agent=observability)
deployment_task = Task(description="Inspect recent deployments.", agent=deployment)
analysis_task = Task(
    description="Synthesize evidence and produce an incident plan.",
    agent=analyst,
    context=[metrics_task, deployment_task],
)

crew = Crew(agents=[observability, deployment, analyst], tasks=[metrics_task, deployment_task, analysis_task])
crew.kickoff()
```


## Compare frameworks

- **CrewAI helps** when the collaboration can be expressed as roles, tasks, and a crew plan.
- **LangGraph helps** when you need explicit state, branches, persistence, checkpoints, and recovery.
- **AutoGen helps** when the learning goal is conversational team coordination and dynamic next-speaker selection.
- **OpenAI Agents SDK helps** when one bounded agent with tools, guardrails, sessions, and tracing is the simplest fit.

The production question stays the same: what does this framework make easier without hiding a control boundary you need to own?

In [ ]:
result["comparison"]


## Reflection

1. Which CrewAI task would you make read-only, and which would require approval before any write action?
2. Where would you add deterministic policy checks around the crew?
3. If this incident were simple service-health lookup, would the crew still be worth the overhead?